# 20 — AWMF-only retrieval with the clinical question distilled from the vignette

This notebook isolates two changes from notebook 18. First, the MedRAG textbook
fallback is removed: retrieval runs over the AWMF German guideline corpus only, so
the result reflects that corpus on its own. Second, before retrieval each MedQA
case is reduced to the single clinical question it is actually asking; that
distilled question drives HyDE, the dense and BM25 searches, and the reranker,
while the full case is still used to write the answer. Everything else — the
strict-grounding prompt, the models, and the four RAGAS metrics — is unchanged, so
the numbers are directly comparable to the earlier notebooks.

The benchmark read here is the model-filtered, balanced set produced by notebook 19.
The corpus-grounded ground truth is set aside; evaluation reports the full 200
against the MedQA reference answer.

## Installs

In [8]:
!pip install -q ragas langchain langchain-openai langchain-huggingface psycopg2-binary pgvector langchain-postgres datasets nest_asyncio sentence-transformers sqlalchemy requests rank_bm25

## Vertex patch

A defensive shim so that importing parts of `langchain_community` does not pull in the Vertex
AI SDK, which is not installed in this environment.

In [9]:
import sys, types
class DummyVertexAI: pass
class DummyChatVertexAI: pass
m = types.ModuleType("langchain_community.llms"); m.VertexAI = DummyVertexAI; sys.modules["langchain_community.llms"] = m
m = types.ModuleType("langchain_community.chat_models"); m.ChatVertexAI = DummyChatVertexAI; sys.modules["langchain_community.chat_models"] = m
m = types.ModuleType("langchain_community.chat_models.vertexai"); m.ChatVertexAI = DummyChatVertexAI; sys.modules["langchain_community.chat_models.vertexai"] = m
m = types.ModuleType("langchain_community.llms.vertexai"); m.VertexAI = DummyVertexAI; sys.modules["langchain_community.llms.vertexai"] = m

## Database, dense store, BM25 index, reranker, models

The AWMF collection is read from the managed database and its BM25 index is rebuilt
from the same chunks. The answer prompt is the same strict-grounding prompt: it
forbids outside knowledge and asks the model to abstain when the passages are
insufficient.

In [10]:
import os, json, time, random
import pandas as pd, torch, numpy as np, nest_asyncio
from google.colab import userdata, drive
from sqlalchemy import create_engine, text
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVector
from langchain_core.prompts import PromptTemplate
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

nest_asyncio.apply()
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/'
df = pd.read_csv(DRIVE_PATH + 'AWMF_Golden_Dataset_200Q_AIFiltered.csv')

NEON = userdata.get('NEON_DATABASE_URL')
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

engine = create_engine(NEON, pool_pre_ping=True, pool_recycle=180, pool_size=2, max_overflow=2,
                       connect_args={"sslmode":"require","keepalives":1,"keepalives_idle":30})

COLLECTION = "awmf_baseline_bge"

# 1. AWMF dense store (read-only) + query embedder
bge = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={'device':'cpu'})
vs = PGVector(embeddings=bge, collection_name=COLLECTION, connection=engine, use_jsonb=True)

# 2. AWMF sparse BM25 index
print("Loading AWMF texts for BM25 index...")
with engine.connect() as conn:
    res = conn.execute(text("""SELECT document FROM langchain_pg_embedding
                               WHERE collection_id = (SELECT uuid FROM langchain_pg_collection WHERE name = :c)"""),
                       {"c": COLLECTION})
    all_texts = [r[0] for r in res]
bm25_retriever = BM25Okapi([doc.lower().split(" ") for doc in all_texts])
print(f"AWMF BM25 index built with {len(all_texts)} chunks.")

# 3. Cross-encoder reranker
print("Loading reranker...")
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cuda" if torch.cuda.is_available() else "cpu")

def make_llm(model, max_tokens=1024):
    return ChatOpenAI(model=model, api_key=os.environ["OPENROUTER_API_KEY"],
                      base_url="https://openrouter.ai/api/v1", temperature=0,
                      max_tokens=max_tokens, max_retries=6, request_timeout=90)

mistral = make_llm("mistralai/mistral-large")

def safe_invoke(llm, prompt, max_tries=8, base=8):
    for a in range(max_tries):
        try:
            return llm.invoke(prompt).content.strip()
        except Exception as e:
            if a == max_tries-1: raise
            w = min(base*(2**a)+random.uniform(0,3), 120)
            print(f"  retry {a+1}: {str(e)[:70]} ... {w:.0f}s"); time.sleep(w)

# HyDE translation, kept as in the hybrid baseline (drives the German AWMF search).
query_gen_prompt = PromptTemplate(template="""You are a German medical expert.
1. First, precisely TRANSLATE the English question into German.
2. Second, write a 2-sentence hypothetical answer to the question in formal German clinical guideline terminology.
Do NOT use bullet points. Output ONLY the German translation followed directly by the German hypothetical answer.

English Question:
{question}""", input_variables=["question"])

# STRICT grounding answer prompt — the lever that raises faithfulness for a medical system.
qa_prompt = PromptTemplate(template="""You are a clinical assistant. Answer the question in ENGLISH using ONLY the reference passages provided below. The passages are German clinical guidelines.

STRICT RULES:
1. Base every statement directly on the passages. Do NOT use any medical knowledge from outside them.
2. Do NOT add background, mechanisms, or details the passages do not state.
3. If the passages do not contain enough information to answer, reply with exactly: NOT_IN_CORPUS
4. Prefer the source's own wording. Be concise and factual.

REFERENCE PASSAGES:
{context}

QUESTION: {question}

ANSWER (English, grounded strictly in the passages):""", input_variables=["context", "question"])
print("Setup complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading AWMF texts for BM25 index...
AWMF BM25 index built with 8687 chunks.
Loading reranker...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Setup complete.


## Retrieval (AWMF-only, distilled query)

Each case is first reduced to its clinical question. That distilled question drives
the German HyDE search over the AWMF store (dense plus BM25), and the cross-encoder
reranks the merged candidates against the same distilled question. No MedRAG corpus
is involved.

In [11]:
# Reduce the vignette to its actual clinical question before retrieving. The case
# is long, but the retrievable fact is a small part of it, so retrieval works on the
# distilled question. The full case is still used to write the answer.
focus_prompt = PromptTemplate.from_template(
    "Restate the single clinical question this case is asking, in one concise sentence using "
    "medical terms. Keep only what is needed to answer it; drop the narrative detail.\n\n"
    "Case:\n{vignette}\n\nClinical question:"
)

def retrieve(question, k_final=10):
    focus_en = (safe_invoke(mistral, focus_prompt.format(vignette=question)) or "").strip() or question
    hyde_de = safe_invoke(mistral, query_gen_prompt.format(question=focus_en))  # German query from the distilled question

    # AWMF (German) candidates: dense in the managed store plus BM25 over the same chunks.
    awmf_dense = [d.page_content for d in vs.as_retriever(search_kwargs={"k": 20}).invoke(hyde_de)]
    awmf_bm25  = bm25_retriever.get_top_n(hyde_de.lower().split(" "), all_texts, n=20)

    # merge + de-duplicate
    pool, seen = [], set()
    for t in awmf_dense + awmf_bm25:
        key = t[:120]
        if key in seen: continue
        seen.add(key); pool.append(t)

    # rerank against the distilled question, not the long vignette
    scores = reranker.predict([[focus_en, t] for t in pool])
    order = sorted(range(len(pool)), key=lambda i: scores[i], reverse=True)[:k_final]
    ctx = [pool[i] for i in order]
    return ctx, focus_en

# Smoke test
_q = df.iloc[0]['English_Open_Question']
_ctx, _focus = retrieve(_q)
print("Vignette:", _q[:90])
print("Distilled question:", _focus)
print("Retrieved", len(_ctx), "AWMF passages. Top passage:", _ctx[0][:200], "...")

Vignette: A 58-year-old woman comes to the physician because of intermittent painful retrosternal du
Distilled question: What is the most likely cause of this patient’s **exertional retrosternal chest pain with associated dyspnea and palpitations**?
Retrieved 10 AWMF passages. Top passage: ▪ Bekannte Herzinsuffizienz (+) 
▪ Bekannter Diabetes mellitus (+) 
▪ Beschwerden sind abhängig von körperlicher Belastung (+) 
▪ Keine Druckempfindlichkeit/Schmerz durch Palpation nicht reproduzierba ...


## Generation loop

Every question is answered from the reranked AWMF passages with the strict-grounding
prompt. The distilled question is stored alongside each item for inspection. Results
are written to Drive after each item so a dropped session resumes cleanly.

In [12]:
work = df.reset_index(drop=True)  # full 200 questions
print(f"Generating grounded answers for {len(work)} questions")

out_file = DRIVE_PATH + "AWMF_ONLY_FOCUS_results.json"
if os.path.exists(out_file):
    res = json.load(open(out_file)); done = set(res["question"])
    print(f"  resuming -- {len(done)} already done")
else:
    res = {"question":[],"focus":[],"answer":[],"contexts":[],"medqa_ground_truth":[]}; done = set()

for i in range(len(work)):
    r = work.iloc[i]; q = r['English_Open_Question']
    if q in done: continue
    try:
        ctx, focus = retrieve(q)
        ans = safe_invoke(mistral, qa_prompt.format(context="\n\n".join(ctx), question=q))

        res["question"].append(q); res["focus"].append(focus); res["answer"].append(ans)
        res["contexts"].append(ctx)
        res["medqa_ground_truth"].append(r['English_Correct_Text'])
        done.add(q)
        json.dump(res, open(out_file, "w"))

        if len(res["question"]) % 10 == 0: print(f"  {len(res['question'])}/{len(work)}")
        time.sleep(1.5)
    except Exception as e:
        print("err", i, str(e)[:100]); time.sleep(8)

print(f"Generation complete -> {out_file}")

Generating grounded answers for 200 questions
  resuming -- 22 already done
  30/200
  40/200
  50/200
  60/200
  70/200
  80/200
  90/200
  100/200
  110/200
  120/200
  130/200
  140/200
  150/200
  160/200
  170/200
  180/200
  190/200
  200/200
Generation complete -> /content/drive/MyDrive/AWMF_ONLY_FOCUS_results.json


## Evaluation

The same GPT-4o-mini judge and the same four RAGAS metrics as the earlier notebooks,
so the numbers stay directly comparable. One view is reported: the full 200 MedQA
questions against the MedQA reference answer.

In [14]:
import os, json
import pandas as pd
from google.colab import userdata, drive
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from datasets import Dataset, Features, Value, Sequence
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
import nest_asyncio

nest_asyncio.apply()
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/'
out_file = DRIVE_PATH + "AWMF_ONLY_FOCUS_results.json"
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

def make_llm(model, max_tokens=4096):
    return ChatOpenAI(model=model, api_key=os.environ["OPENROUTER_API_KEY"],
                      base_url="https://openrouter.ai/api/v1", temperature=0,
                      max_tokens=max_tokens, max_retries=6, request_timeout=90)

print("Loading embedding model for Ragas...")
bge = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={'device':'cpu'})

judge = LangchainLLMWrapper(make_llm("openai/gpt-4o-mini", max_tokens=4096))
remb = LangchainEmbeddingsWrapper(bge)
feat = Features({"question":Value("string"),"answer":Value("string"),
                 "contexts":Sequence(Value("string")),"ground_truth":Value("string")})

def score(path, gt_key):
    d = json.load(open(path))
    dd = {"question":d["question"], "answer":d["answer"], "contexts":d["contexts"],
          "ground_truth":d[gt_key]}
    ds = Dataset.from_dict(dd, features=feat)
    out = evaluate(ds, metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
                   llm=judge, embeddings=remb, run_config=RunConfig(timeout=300, max_workers=2, max_retries=5))
    return len(d["question"]), dict(out.to_pandas()[["context_precision","context_recall","faithfulness","answer_relevancy"]].mean().round(3))

print("=== AWMF-ONLY, DISTILLED QUERY ===")
n, s = score(out_file, "medqa_ground_truth")
print(f"Full 200 (MedQA GT, n={n}): ", s)

/tmp/ipykernel_3552/2018752458.py:8: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_3552/2018752458.py:8: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_3552/2018752458.py:8: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import context_precis

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading embedding model for Ragas...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

/tmp/ipykernel_3552/2018752458.py:28: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge = LangchainLLMWrapper(make_llm("openai/gpt-4o-mini", max_tokens=4096))
/tmp/ipykernel_3552/2018752458.py:29: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  remb = LangchainEmbeddingsWrapper(bge)


=== AWMF-ONLY, DISTILLED QUERY ===


Evaluating:   0%|          | 0/808 [00:00<?, ?it/s]

Full 200 (MedQA GT, n=202):  {'context_precision': np.float64(0.288), 'context_recall': np.float64(0.33), 'faithfulness': np.float64(0.595), 'answer_relevancy': np.float64(0.203)}
